In [1]:
import os
if os.path.basename(os.getcwd()) == 'Demonstrations':
    os.chdir('..')

<div style="background-color: #51daca; color: white; padding: 30px; border-radius: 0px;">
<h1 style="margin: 0;  color: #04335a">Convergence Test for the Stokes Equations </h3>
</div>

Imports:

In [2]:
from scipy.sparse.linalg import spsolve
import pandas as pd
import numpy as np

from Utilities.Stokes_felib import *
from Utilities.Mesh_processing import *
from Solvers.Exchanger_Device import compute_U_P_solution

<div style="background-color: #7ac4ef; color: white; padding: 10px; border-radius: 0px;">
<h3 style="margin: 0; color: #11116e">Convergence Analysis via Reference Solutions on an Exchanger Device</h3>
</div>

Reference Solution computation:

In [3]:
data = np.load('Meshes/exchanger_device_altered_mesh_data.npz')
p_raw = data['p']
tri_idx = data['t_raw']
data.close()

p_base, e_base, t_base = build_stable_mesh(p_raw, tri_idx)
p_coarse_ref, e_coarse_ref, t_coarse_ref = refine_n_times(p_base, e_base, t_base, number_of_refinements=4)
p_fine_ref, e_fine_ref, t_fine_ref = refine(p_coarse_ref, e_coarse_ref, t_coarse_ref)

In [4]:
ux_ref, uy_ref, p_ref = compute_U_P_solution(p_fine_ref, t_fine_ref, e_fine_ref, p_coarse_ref, t_coarse_ref)
reference_solution = [ux_ref, uy_ref, p_ref]

save_simulation_data(p_fine_ref, e_fine_ref, t_fine_ref,
                     p_coarse_ref, e_coarse_ref, t_coarse_ref,
                     ux_ref, uy_ref, p_ref,
                     name='Reference_Solution')

K is of the shape (867059, 867059)
Solving lifted system...
||div|| = 0.12080833829349444
max div = 0.015190971316770049
Simulation 'Reference_Solution' data saved.


Error table generation:

In [ ]:
p_original, e_original, t_original = build_stable_mesh(p_raw, tri_idx)

for i in range(0,4):
    p_coarse, e_coarse, t_coarse = refine_n_times(p_original, e_original, t_original, 
                                                  number_of_refinements=i)
    p_fine, e_fine, t_fine = refine(p_coarse, e_coarse, t_coarse)
    ux, uy, p_sol = compute_U_P_solution(p_fine, t_fine, e_fine, p_coarse, t_coarse)

    save_simulation_data(p_fine, e_fine, t_fine,
                         p_coarse, e_coarse, t_coarse,
                         ux, uy, p_sol,
                         name=f'Convergence_Step_{i}')

In [6]:
# 1. Load your highly refined reference solution file once
p_fine_ref, _, _, p_coarse_ref, _, _, ux_ref, uy_ref, p_ref = load_simulation_data('Solutions/Reference_Solution.npz')

results_list = []

# 2. Pure data loading loop (Levels 0, 1, 2, 3 matching your saved files)
saved_levels = [0, 1, 2, 3]

for i in saved_levels:
    # Just load the files directly from disk
    p_fine, e_fine, t_fine, p_coarse, e_coarse, t_coarse, ux, uy, p_sol = load_simulation_data(f'Solutions/Convergence_Step_{i}.npz')
    
    # Project reference solution coordinates onto this step's nodes
    ux_ref_interp = griddata((p_fine_ref[:, 0], p_fine_ref[:, 1]), ux_ref, (p_fine[:, 0], p_fine[:, 1]), method='cubic', fill_value=0.0)
    uy_ref_interp = griddata((p_fine_ref[:, 0], p_fine_ref[:, 1]), uy_ref, (p_fine[:, 0], p_fine[:, 1]), method='cubic', fill_value=0.0)
    p_ref_interp  = griddata((p_coarse_ref[:, 0], p_coarse_ref[:, 1]), p_ref, (p_coarse[:, 0], p_coarse[:, 1]), method='cubic', fill_value=0.0)

    Nv = p_fine.shape[0]
    Np = p_coarse.shape[0]

    # Raw Euclidean vector differences
    raw_err_ux = np.linalg.norm(ux - ux_ref_interp)
    raw_err_uy = np.linalg.norm(uy - uy_ref_interp)
    raw_err_u  = np.sqrt(raw_err_ux**2 + raw_err_uy**2)
    raw_err_p  = np.linalg.norm(p_sol - p_ref_interp)

    # Scale by spatial coordinate node density to convert to physical L2 error equivalents
    l2_err_u = raw_err_u / np.sqrt(Nv)
    l2_err_p = raw_err_p / np.sqrt(Np)

    results_list.append({
        'Level': i,
        'Mesh Size': f'h0/{2**i}',
        'Velocity DOFs': 2 * Nv,
        'Pressure DOFs': Np,
        'L2 Error u': l2_err_u,
        'L2 Error p': l2_err_p
    })

# 3. Process data into a clean, classic academic presentation table
df = pd.DataFrame(results_list)

# Compute proper algebraic rate differences between refinement steps
df['Rate (ru)'] = np.log(df['L2 Error u'].shift(1) / df['L2 Error u']) / np.log(2)
df['Rate (rp)'] = np.log(df['L2 Error p'].shift(1) / df['L2 Error p']) / np.log(2)

# Apply formatting cleanups for clear terminal/notebook print alignment
df['L2 Error u'] = df['L2 Error u'].map('{:.4e}'.format)
df['L2 Error p'] = df['L2 Error p'].map('{:.4e}'.format)
df['Rate (ru)'] = df['Rate (ru)'].map('{:.3f}'.format).replace('nan', '—')
df['Rate (rp)'] = df['Rate (rp)'].map('{:.3f}'.format).replace('nan', '—')

In [7]:
print("\n" + "="*95)
print("                           STOKES FEM CONVERGENCE TABLE (LOADED SOLUTIONS)")
print("="*95)
print(df.to_string(index=False))
print("="*95)


                           STOKES FEM CONVERGENCE TABLE (LOADED SOLUTIONS)
 Level Mesh Size  Velocity DOFs  Pressure DOFs L2 Error u L2 Error p Rate (ru) Rate (rp)
     0      h0/1           3262            442 1.1575e-01 1.7831e+00         —         —
     1      h0/2          12506           1631 6.7977e-02 9.8238e-01     0.768     0.860
     2      h0/4          48946           6253 4.3875e-02 4.9816e-01     0.632     0.980
     3      h0/8         193634          24473 3.1974e-02 2.3816e-01     0.457     1.065
